In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from collections import Counter
import scipy.sparse as sp

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import time

# Подготавливаем датасет для загрузки

In [ ]:
data = pd.read_pickle("text_data.pkl")

Забираем 5% записей (чтобы нейросеть обучилась быстрее)

In [ ]:
data = data.sample(frac = 0.05)

Собираем все слова из предложений

In [ ]:
all_words = [word.strip() for sentence in data['sentence'] for word in sentence if word.strip() != ""]
all_words = sorted(all_words)

Считаем частоту слов

In [ ]:
word_freq = Counter(all_words)

Создаем словарь из слов с частотой выше указанной

In [ ]:
K = 3  # порог
vocabulary = [word for word, freq in word_freq.items() if freq >= K]

Добавляем специальные символы для пустоты и неизвестного слова

In [ ]:
vocabulary = ['<PAD>', '<UNK>'] + vocabulary

Считаем размер словаря

In [ ]:
vocab_size = len(vocabulary)
print("В словаре :", vocab_size, f"слов (частота ≥ {K})")

В словаре : 1134 слов (частота ≥ 3)


Находим максимальную длину предложения

In [ ]:
data['sentence_len'] = data['sentence'].apply(lambda x: len(x))

Определяем оптимальный размер длины последовательности для передачи в нейросеть

In [ ]:
data['sentence_len'].describe()

,sentence_len
count,1526.000000
mean,15.587811
std,13.233533
min,0.000000
25%,6.000000
50%,12.000000
75%,21.000000
max,99.000000


In [ ]:
max_length = data['sentence'].apply(lambda x: len(x)).max()
print("Максимально длинное предложение состоит из :", max_length, "слов")

Максимально длинное предложение состоит из : 99 слов


In [ ]:
print("95% квантиль длины предложений: ", data['sentence_len'].quantile(0.95))

95% квантиль длины предложений:  41.0


Берем 95% квантиль в качестве длины последовательности для нейросети

In [ ]:
max_length = int(data['sentence_len'].quantile(0.95))

Создаем словарь перевода слова в индексы (порядковые номера в словаре)

In [ ]:
for i, item in enumerate(['a', 'b', 'c', 'd']):
  print(i, item)

0 a
1 b
2 c
3 d


In [ ]:
word_to_index = {word: idx for idx, word in enumerate(vocabulary)}
vocab_size = len(vocabulary)
print("Длина словаря: ", vocab_size)

Длина словаря:  1134


Переводим в индексы

In [ ]:
def sentence_to_index(sentence, max_len):
  # Преобразуем предложение в последовательность индексов
  indices = [word_to_index[word.lower()] if word.lower() in vocabulary else word_to_index['<UNK>'] for word in sentence ]

  # Пэддинг до max_len
  if len(indices) < max_len:
      indices += [word_to_index['<PAD>']] * (max_len - len(indices))
  else:
      indices = indices[:max_len]  # обрезаем, если длиннее

  return indices

In [ ]:
data['indices'] = data['sentence'].apply(sentence_to_index, args = (max_length, ))

In [ ]:
data.head()

,sentence,author,sentence_len,indices
28308,"[вот, мы, папаше, покажем, какие, мы, молодцы,...",Толстой,34,"[123, 519, 1, 1, 380, 519, 1, 345, 1, 666, 1, ..."
5080,"[очень, ответила, дуня]",Достоевский,3,"[655, 1, 282, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
21146,[завтра],Толстой,1,"[321, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0..."
25894,"[у, нас, на, постирушечки, две, женщины, прист...",Толстой,12,"[1033, 540, 522, 1, 218, 311, 1, 1, 2, 1, 135,..."
4058,"[довольно, произнес, он, решительно, и, торжес...",Достоевский,13,"[249, 772, 626, 815, 345, 1, 780, 1, 780, 1, 1..."


Переводим в One-Hot

In [ ]:
def sentence_to_one_hot(sentence, max_len):
  # Преобразуем предложение в последовательность индексов
  indices = [word_to_index[word.lower()] if word.lower() in vocabulary else word_to_index['<UNK>'] for word in sentence ]

  # Пэддинг до max_len
  if len(indices) < max_len:
      indices += [word_to_index['<PAD>']] * (max_len - len(indices))
  else:
      indices = indices[:max_len]  # обрезаем, если длиннее

  # Создаём one‑hot матрицу (max_len, vocab_size)
  one_hot = np.zeros((max_len, vocab_size), dtype = np.float32)
  for i, idx in enumerate(indices):
      one_hot[i, idx] = 1.0

  return one_hot

In [ ]:
data['one_hot'] = data["sentence"].apply(sentence_to_one_hot, args = (max_length, ))
data['target'] = np.where(data["author"] == "Достоевский", 0, 1)

In [ ]:
data.head()

,sentence,author,sentence_len,indices,one_hot,target
28308,"[вот, мы, папаше, покажем, какие, мы, молодцы,...",Толстой,34,"[123, 519, 1, 1, 380, 519, 1, 345, 1, 666, 1, ...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",1
5080,"[очень, ответила, дуня]",Достоевский,3,"[655, 1, 282, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",0
21146,[завтра],Толстой,1,"[321, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",1
25894,"[у, нас, на, постирушечки, две, женщины, прист...",Толстой,12,"[1033, 540, 522, 1, 218, 311, 1, 1, 2, 1, 135,...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",1
4058,"[довольно, произнес, он, решительно, и, торжес...",Достоевский,13,"[249, 772, 626, 815, 345, 1, 780, 1, 780, 1, 1...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",0


In [ ]:
data['target'].value_counts()

,count
target,
1,914
0,612


На самом деле, хранить One Hot в pandas dataframe крайне расточительно, т.к. датасет из 7 Мб в виде текста превратился в более чем 7Гб в виде разреженных матриц.

Для работы с разреженными матрицами разумнее использовать другой подход

In [ ]:
def sentences_to_sparse_onehot(sentence, word_to_index, max_len):
    indices = []
    for pos, word in enumerate(sentence):
        if pos >= max_len:
            break
        if word in word_to_index:
            indices.append((pos, word_to_index[word]))

    # Создаём sparse-матрицу в формате COO
    if not indices:
        # Если все слова редкие — возвращаем нулевую матрицу
        return sp.coo_matrix((max_len, vocab_size), dtype=np.float32)
    row, col = zip(*indices)
    data = np.ones(len(row), dtype=np.float32)
    return sp.coo_matrix((data, (row, col)), shape=(max_len, vocab_size))

# Кодируем все предложения
X_sparse = [sentences_to_sparse_onehot(sentence, word_to_index, max_length) for sentence in data['sentence']]
y = torch.tensor(data['target'].values, dtype=torch.float32)

# Делим на обучающую и тестовую выборки

In [ ]:
train_indices, test_indices = train_test_split(
    range(len(X_sparse)),
    test_size=0.3,
    random_state=42,
    stratify=data['target'].values
)

In [ ]:
# Формируем train и test наборы
X_train = [X_sparse[i] for i in train_indices]
y_train = torch.tensor(data['target'].iloc[train_indices].values, dtype=torch.float32)

X_test = [X_sparse[i] for i in test_indices]
y_test = torch.tensor(data['target'].iloc[test_indices].values, dtype=torch.float32)

print(f"Размер train: {len(X_train)}")
print(f"Размер test: {len(X_test)}")

Размер train: 1068
Размер test: 458


# Создаем DataSet и DataLoader

In [ ]:
class SparseOneHotDataset(Dataset):
    def __init__(self, sparse_data, labels):
        self.sparse_data = sparse_data
        self.labels = labels

    def __len__(self):
        return len(self.sparse_data)

    def __getitem__(self, idx):
        # Преобразуем scipy.sparse в torch.sparse
        coo = self.sparse_data[idx].tocoo()
        values = torch.tensor(coo.data, dtype=torch.float32)
        indices = torch.stack([
            torch.tensor(coo.row, dtype=torch.long),
            torch.tensor(coo.col, dtype=torch.long)
        ])
        sparse_tensor = torch.sparse_coo_tensor(
            indices, values, size=coo.shape
        ).coalesce()
        return sparse_tensor, self.labels[idx]

In [ ]:
def sparse_collate_fn(batch):
    """
    Объединяет список (sparse_tensor, label) в батч.
    - sparse_tensor: список torch.sparse_coo_tensor → объединяем в один sparse-батч
    - label: список тензоров → объединяем в один плотный тензор
    """
    # Разделяем данные и метки
    sparse_tensors, labels = zip(*batch)

    # Объединяем метки в плотный тензор
    labels = torch.stack(labels)

    # Собираем индексы и значения для единого sparse-батча
    indices_list = []
    values_list = []
    batch_size = len(sparse_tensors)
    seq_len, vocab_size = sparse_tensors[0].size()


    for i, sparse_tensor in enumerate(sparse_tensors):
        # Получаем координаты (i, j) и значения
        indices = sparse_tensor.indices()
        values = sparse_tensor.values()

        # Добавляем номер образца в батчи (новая ось)
        batch_indices = torch.full((1, indices.size(1)), i, dtype=torch.long, device=indices.device)
        full_indices = torch.cat([batch_indices, indices], dim=0)  # shape: (3, nnz)


        indices_list.append(full_indices)
        values_list.append(values)

    # Конкатенируем все индексы и значения
    all_indices = torch.cat(indices_list, dim=1)   # (3, total_nnz)
    all_values = torch.cat(values_list)           # (total_nnz,)

    # Создаём единый sparse-тензор для батча
    batch_sparse = torch.sparse_coo_tensor(
        all_indices,
        all_values,
        size=(batch_size, seq_len, vocab_size)
    ).coalesce()

    return batch_sparse, labels

In [ ]:
# Создаём datasets
train_dataset = SparseOneHotDataset(X_train, y_train)
test_dataset = SparseOneHotDataset(X_test, y_test)

# DataLoaders
train_dataloader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=sparse_collate_fn  # указываем нашу функцию
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=sparse_collate_fn  # то же самое для test
)

# Создаем ручную RNN

In [ ]:
class SparseRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super(SparseRNN, self).__init__()
        self.hidden_size = hidden_size

        # Веса: вход (one-hot) → скрытое состояние
        self.W_ih = nn.Linear(vocab_size, hidden_size, bias=False)
        self.W_hh = nn.Linear(hidden_size, hidden_size)
        self.activation = nn.Tanh()

        # Выход: скрытое → логит (бинарная классификация)
        self.fc = nn.Linear(hidden_size, 1)  # 1 выход для бинарной классификации (не 2 класса, как это было бы в мультиклассовой задаче)


    def forward(self, x_sparse):
        batch_size = x_sparse.size(0)
        seq_len = x_sparse.size(1)

        h = torch.zeros(batch_size, self.hidden_size).to(x_sparse.device)

        for t in range(seq_len):
            # Вырезаем временной срез t (sparse)
            x_t_sparse = x_sparse[:, t, :]  # (batch_size, vocab_size) sparse
            # Умножение sparse на dense (W_ih)
            x_dense = torch.sparse.mm(x_t_sparse, self.W_ih.weight.t())
            h = self.activation(x_dense + self.W_hh(h))

        logits = self.fc(h)  # (batch_size, 1)
        return logits.squeeze(-1)  # (batch_size,) для BCEWithLogitsLoss

model = SparseRNN(vocab_size=vocab_size, hidden_size=64)

# Обучаем рукописную RNN

In [ ]:
LEARNING_RATE = 0.0001
EPOCHS = 10
DEVICE = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

model = SparseRNN(vocab_size = vocab_size, hidden_size = 64)
criterion = nn.BCEWithLogitsLoss() # Функция потерь для бинарной классификации
optimizer = torch.optim.Adam(model.parameters(), lr = LEARNING_RATE)

Обучаем модель

In [ ]:
start_time = time.time()
model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for X_batch, y_batch in train_dataloader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_dataloader):.4f}")
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Время обучения: {elapsed_time:.4f} секунд")

Epoch 1, Loss: 0.6815
Epoch 2, Loss: 0.6743
Epoch 3, Loss: 0.6740
Epoch 4, Loss: 0.6731
Epoch 5, Loss: 0.6716
Epoch 6, Loss: 0.6713
Epoch 7, Loss: 0.6697
Epoch 8, Loss: 0.6604
Epoch 9, Loss: 0.6391
Epoch 10, Loss: 0.5738
Время обучения: 67.8358 секунд


Сохраняем модель

In [ ]:
torch.save(model.state_dict(), "rnn_model.pth")

Загружаем модель

In [ ]:
model = SparseRNN(vocab_size = vocab_size, hidden_size = 64)
model.load_state_dict(torch.load("rnn_model.pth", weights_only = True))

Применение к тестовому датасету

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

model.eval()  # переводим модель в режим inference

# Собираем истинные метки и предсказания
y_true = []
y_pred = []

with torch.no_grad():
    for X_batch, y_batch in test_dataloader:
        # Получаем логиты (необработанные выходы)
        logits = model(X_batch)
        # Преобразуем в вероятности через сигмоиду
        probs = torch.sigmoid(logits)
        # Бинаризация: порог 0.5
        predicted = (probs > 0.5).float()
        # Сохраняем для метрик
        y_true.extend(y_batch.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

# Конвертируем в numpy-массивы
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Вычисляем метрики
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

print(f"Test Accuracy:  {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall:    {recall:.4f}")


Test Accuracy:  0.5721
Test Precision: 0.6309
Test Recall:    0.6861


In [ ]:
y_test.mean()

tensor(0.5983)